In [1]:
! ollama list

NAME                                                 ID              SIZE      MODIFIED     
ORANSight_Qwen_7B_Instruct:latest                    ab11d0119d6a    15 GB     2 hours ago     
ORANSight_Qwen_3B_Instruct:latest                    d578a3f030a6    6.2 GB    2 hours ago     
ORANSight_Qwen_1.5B_Instruct:latest                  07bc24db663e    3.1 GB    3 hours ago     
phi:latest                                           e2fd6321a5fe    1.6 GB    20 hours ago    
tinyllama:latest                                     2644915ede35    637 MB    20 hours ago    
hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest    98c4eb4a3287    68 MB     6 days ago      
mxbai-embed-large:latest                             468836162de7    669 MB    5 weeks ago     


In [7]:
import csv
import json
import time
import requests

# --- CONFIGURATION ---
INPUT_FILE = "QwenTesting/Questions.csv"
OUTPUT_FILE = "results.csv"
MODEL_NAME = "ORANSight_Qwen_1.5B_Instruct:latest" # Variable to easily switch models
OLLAMA_URL = "http://localhost:11434/api/generate"

def run_evaluation():
    results = []

    # 1. Read the input file
    try:
        with open(INPUT_FILE, mode='r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
    except FileNotFoundError:
        print(f"Error: {INPUT_FILE} not found.")
        return

    total_questions = len(lines)
    print(f"Starting evaluation with model: {MODEL_NAME}...")
    print(f"Total questions to process: {total_questions}\n")

    for index, line in enumerate(lines, start=1):
        try:
            # Parsing the line (Format: ["Question", ["Opt1", "Opt2"...], "CorrectIndex"])
            data = json.loads(line)
            question = data[0]
            options = "\n".join(data[1])
            correct_idx = data[2]

            # Construct the prompt
            prompt = f"{question}\n\nOptions:\n{options}\n\nRespond with ONLY the number of the correct option (e.g., 1, 2, 3, or 4)."

            # 2. Measure Latency and Call Ollama
            payload = {
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0} # Set to 0 for consistent evaluation
            }

            start_time = time.time()
            response = requests.post(OLLAMA_URL, json=payload)
            response.raise_for_status()
            end_time = time.time()
            
            latency = end_time - start_time
            model_answer = response.json().get("response", "").strip()
            
            results.append({
                "question": question,
                "model_answer": model_answer,
                "correct_answer": correct_idx,
                "latency_sec": round(latency, 4)
            })

            # Progress Logger
            print(f"Question [{index} of {total_questions}] success")

        except Exception as e:
            print(f"Question [{index} of {total_questions}] failed: {e}")

    # 3. Write to Output CSV
    keys = ["question", "model_answer", "correct_answer", "latency_sec"]
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(results)

    print(f"\nEvaluation complete. Results saved to {OUTPUT_FILE}")

# Execute
run_evaluation()

Starting evaluation with model: ORANSight_Qwen_1.5B_Instruct:latest...
Total questions to process: 1139

Question [1 of 1139] success
Question [2 of 1139] success
Question [3 of 1139] success
Question [4 of 1139] success
Question [5 of 1139] success
Question [6 of 1139] success
Question [7 of 1139] success
Question [8 of 1139] success
Question [9 of 1139] success
Question [10 of 1139] success
Question [11 of 1139] success
Question [12 of 1139] success
Question [13 of 1139] success
Question [14 of 1139] success
Question [15 of 1139] success
Question [16 of 1139] success
Question [17 of 1139] success
Question [18 of 1139] success
Question [19 of 1139] success
Question [20 of 1139] success
Question [21 of 1139] success
Question [22 of 1139] success
Question [23 of 1139] success
Question [24 of 1139] success
Question [25 of 1139] success
Question [26 of 1139] success
Question [27 of 1139] success
Question [28 of 1139] success
Question [29 of 1139] success
Question [30 of 1139] success
Ques

In [ ]:
import pandas as pd

def extract_data(file_path):
    # Load the CSV, ensuring it handles multiline fields correctly
    df = pd.read_csv(file_path, quotechar='"', skipinitialspace=True)
    
    # Extract only the columns of interest
    # We use .copy() to avoid SettingWithCopy warnings if we modify it later
    extracted_df = df[['model_answer', 'latency_sec']].copy()
    
    # Display the first few results
    print("Extracted Data (First 5 rows):")
    print(extracted_df.head())
    
    # Calculate average latency if needed
    avg_latency = extracted_df['latency_sec'].mean()
    print(f"\nAverage Latency: {avg_latency:.4f} sec")
    
    return extracted_df

# To run the script:
extracted_data('results.csv')